In [14]:
# ── CELLULE 1 : Chargement des données ──
import pandas as pd
import os, csv
from sklearn.preprocessing import LabelEncoder

Proj = "PRJNA755688-stage12"
path = "/mnt/g/ngs/data/" + Proj
path1 = path + "/"
path2 = path + "/rna/"
fichiers_non_trouve = []

f = path1 + "liste-files.csv"
df_tot = pd.DataFrame()

with open(f, 'r') as csvfile:
    filereader = csv.reader(csvfile, delimiter=',')
    for row in filereader:
        sample = row[0]
        label = row[1]

        f1 = path2 + "results-" + sample + ".txt"
        if os.path.isfile(f1):
            df = pd.read_csv(f1, sep="\t")
            df["sample"] = sample
            df_pivot = df.pivot_table(values=['count'], columns='RNA', index=['sample'])

            if "I" in label:
                label = "II"
            if "healthy" in label:
                label = "control"
            if "Normal" in label:
                label = "control"

            df_pivot["label"] = label
            df_tot = pd.concat([df_tot, df_pivot])
        else:
            fichiers_non_trouve.append(f1)

print("Fichiers non trouvés :", len(fichiers_non_trouve))
#print(df_tot.head())

encoder = LabelEncoder()
#print(df_tot.label.unique())

print(df_tot.label.value_counts())

target = encoder.fit_transform(df_tot.label)
df_tot = df_tot.fillna(0)

Fichiers non trouvés : 0
label
control    561
II         399
Name: count, dtype: int64


In [15]:
# ── CELLULE 2 : Vérification labels ──
print(df_tot["label"])

sample
SRR17005923         II
SRR17005924         II
SRR17005930         II
SRR17005931         II
SRR17005945         II
                ...   
SRR16919803    control
SRR16919804    control
SRR16919805    control
SRR16919806    control
SRR16919807    control
Name: label, Length: 960, dtype: object


In [3]:
# ── CELLULE 3 : Vérification target ──
print(target)
print(target.shape)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 

In [16]:
# ── CELLULE 4 : Split train/test + équilibrage sur train UNIQUEMENT ──
from sklearn.model_selection import train_test_split

# Split stratifié AVANT équilibrage (évite data leakage)
labels = df_tot["label"].copy()
idx_train, idx_test = train_test_split(
    df_tot.index, test_size=0.20, random_state=222, stratify=labels
)

df_train_all = df_tot.loc[idx_train].copy()
y_train_all = labels.loc[idx_train]
df_test_all = df_tot.loc[idx_test].copy()
y_test_all = labels.loc[idx_test]

# Sauvegarder le test set pour évaluation future
X_test_final = df_test_all.reset_index(drop=True)
y_test_final = y_test_all.reset_index(drop=True)

# Équilibrer UNIQUEMENT le train (undersampling majoritaire)
class_counts = y_train_all.value_counts()
min_count = class_counts.min()

df_train_balanced = pd.DataFrame()
for cls in class_counts.index:
    df_cls = df_train_all[y_train_all == cls]
    if len(df_cls) > min_count:
        df_cls = df_cls.sample(n=min_count, random_state=42)
    df_train_balanced = pd.concat([df_train_balanced, df_cls])

df_tot = df_train_balanced.reset_index(drop=True)
target = encoder.fit_transform(df_tot["label"])

print("Distribution train après équilibrage :")
print(df_tot["label"].value_counts())
print(f"Train équilibré: {df_tot.shape}, Test réservé: {X_test_final.shape}")
print("[OK] Pas de data leakage : split avant équilibrage")

Distribution train après équilibrage :
label
control    319
II         319
Name: count, dtype: int64
Train équilibré: (638, 2653), Test réservé: (303, 2653)
[OK] Pas de data leakage : split avant équilibrage


In [17]:
# ── CELLULE 5 : Suppression colonne label ──
if "label" in df_tot.columns:
    df_tot = df_tot.drop(columns=["label"])
print(f"df_tot shape (sans label): {df_tot.shape}")

df_tot shape (sans label): (638, 2652)


In [18]:
# ── CELLULE 6 : Vérification shapes ──
print(f"df_tot shape (prêt pour ML): {df_tot.shape}")
print(f"target shape: {target.shape}")

df_tot shape (prêt pour ML): (638, 2652)
target shape: (638,)


In [19]:
# ── CELLULE 7 : Diagnostic + filtrage des RNAs ──
import numpy as np

print("=" * 60)
print("DIAGNOSTIC AVANT FILTRAGE")
print("=" * 60)
print(f"Nb features (RNAs): {df_tot.shape[1]}")
print(f"Nb échantillons:      {df_tot.shape[0]}")

zero_pct = (df_tot == 0).sum().sum() / (df_tot.shape[0] * df_tot.shape[1]) * 100
print(f"% de valeurs = 0:      {zero_pct:.1f}%")

nonzero_pct = (df_tot > 0).sum(axis=0) / df_tot.shape[0]
exprimes = (nonzero_pct > 0.20).sum()
# print(f"RNAs exprimés dans >20%: {exprimes}/{df_tot.shape[1]}")

variances = df_tot.var(axis=0).sort_values(ascending=False)
# print(f"\nTop 5 RNAs (variance max):")
for rank, (idx, var) in enumerate(variances.head(5).items(), 1):
    name = idx[1] if isinstance(idx, tuple) else idx
    # print(f"  {rank}. {name}: {var:.2e}")

print("\n" + "=" * 60)
print("FILTRAGE")
print("=" * 60)

mask_exprime = nonzero_pct >= 0.20
df_tot_f1 = df_tot.loc[:, mask_exprime]
print(f"Étape 1 (exprimés ≥20%): {df_tot.shape[1]} → {df_tot_f1.shape[1]}")

MAX_FEATURES = 50
if df_tot_f1.shape[1] > MAX_FEATURES:
    variances_f1 = df_tot_f1.var(axis=0).sort_values(ascending=False)
    top_features = variances_f1.head(MAX_FEATURES).index
    df_tot_filtered = df_tot_f1[top_features].copy()
    print(f"Étape 2 (top {MAX_FEATURES}): {df_tot_f1.shape[1]} → {df_tot_filtered.shape[1]}")
else:
    df_tot_filtered = df_tot_f1.copy()

print(f"Features retirées: {df_tot.shape[1] - df_tot_filtered.shape[1]}")
# print(f"Top 10 RNAs retenus:")
for rank, idx in enumerate(variances_f1.head(10).index, 1):
    name = idx[1] if isinstance(idx, tuple) else idx
    # print(f"  {rank}. {name}")
print("[OK] Prêt pour AutoGluon")

DIAGNOSTIC AVANT FILTRAGE
Nb features (RNAs): 2652
Nb échantillons:      638
% de valeurs = 0:      69.6%

FILTRAGE
Étape 1 (exprimés ≥20%): 2652 → 2097
Étape 2 (top 50): 2097 → 50
Features retirées: 2602
[OK] Prêt pour AutoGluon


In [20]:
# ── CELLULE 8 : AutoGluon avec PCA (medium_quality) ──
from autogluon.tabular import TabularPredictor
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import numpy as np
import warnings
warnings.filterwarnings("ignore")

try:
    X = df_tot_filtered.copy()
    print("[INFO] Données filtrées")
except NameError:
    X = df_tot.copy()
    print("[WARN] Données NON filtrées")

if isinstance(X.columns, pd.MultiIndex):
    X.columns = ['_'.join(str(c) for c in col) for col in X.columns]

y = encoder.inverse_transform(target)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=222)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# NCA+PCA fit sur TRAIN uniquement
nca = NeighborhoodComponentsAnalysis(random_state=42)
X_train_nca = nca.fit_transform(X_train, encoder.transform(y_train))
X_test_nca = nca.transform(X_test)
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_nca)
X_test_pca = pca.transform(X_test_nca)
print(f"Composantes PCA: {pca.n_components_}")

df_train = pd.DataFrame(X_train_pca)
df_train.columns = df_train.columns.astype(str)
df_train["label"] = y_train
df_test = pd.DataFrame(X_test_pca)
df_test.columns = df_test.columns.astype(str)
df_test["label"] = y_test
print(f"Train PCA: {df_train.shape}, Test PCA: {df_test.shape}")

predictor = TabularPredictor(label="label", problem_type="multiclass", eval_metric="accuracy", verbosity=2).fit(
    train_data=df_train, time_limit=300, presets="medium_quality")

print("\n" + "=" * 60)
print("LEADERBOARD (PCA)")
print("=" * 60)
leaderboard = predictor.leaderboard(df_test, silent=True)
print(leaderboard)

print("\n" + "=" * 60)
print("ÉVALUATION")
print("=" * 60)
perf = predictor.evaluate(df_test, auxiliary_metrics=True, silent=True)
for k, v in perf.items():
    print(f"  {k}: {v:.4f}")

print("\n" + "=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)
importance = predictor.feature_importance(df_test)
print(importance.head(10))

No path specified. Models will be saved in: "AutogluonModels/ag-20260610_150041"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #5262-Microsoft Fri Jan 01 08:00:00 PST 2016
CPU Count:          16
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       2.68 GB / 23.91 GB (11.2%)
Disk Space Avail:   414.57 GB / 953.85 GB (43.5%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'


[INFO] Données filtrées
Train: (510, 50), Test: (128, 50)
Composantes PCA: 3
Train PCA: (510, 4), Test PCA: (128, 4)


Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/mnt/g/ngs/scripts/AutogluonModels/ag-20260610_150041"
Train Data Rows:    510
Train Data Columns: 3
Label Column:       label
Problem Type:       multiclass
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Train Data Class Count: 2
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    2738.98 MB
	Train Data (Original)  Memory Usage: 0.01 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator..


LEADERBOARD (PCA)


Computing feature importance via permutation shuffling for 3 features using 128 rows with 5 shuffle sets...
	0.28s	= Expected runtime (0.06s per shuffle set)
	0.12s	= Actual runtime (Completed 5 of 5 shuffle sets)


                  model  score_test  score_val eval_metric  pred_time_test  \
0              LightGBM    1.000000        1.0    accuracy        0.001057   
1         LightGBMLarge    1.000000        1.0    accuracy        0.001673   
2            LightGBMXT    1.000000        1.0    accuracy        0.001966   
3              CatBoost    1.000000        1.0    accuracy        0.003111   
4        NeuralNetTorch    1.000000        1.0    accuracy        0.007467   
5               XGBoost    1.000000        1.0    accuracy        0.008208   
6      RandomForestEntr    1.000000        1.0    accuracy        0.077879   
7        ExtraTreesEntr    1.000000        1.0    accuracy        0.086622   
8        ExtraTreesGini    1.000000        1.0    accuracy        0.087197   
9      RandomForestGini    1.000000        1.0    accuracy        0.245187   
10      NeuralNetFastAI    0.992188        1.0    accuracy        0.011889   
11  WeightedEnsemble_L2    0.992188        1.0    accuracy      

In [22]:
# ── CELLULE 9 : AutoGluon sans PCA (medium_quality, vrai test set) ──
from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings("ignore")

# Utiliser df_tot_filtered si disponible, sinon df_tot (train équilibré)
try:
    df_ml = df_tot_filtered.copy()
    print("[INFO] Données filtrées")
except NameError:
    df_ml = df_tot.copy().drop(columns=["label"])

if isinstance(df_ml.columns, pd.MultiIndex):
    df_ml.columns = ['_'.join(str(c) for c in col) for col in df_ml.columns]

df_ml["label"] = encoder.inverse_transform(target)
df_ml = df_ml.reset_index(drop=True)

# Test set = celui réservé en cellule 4 (pas de split refait)
try:
    X_test = X_test_final.copy()
    if isinstance(X_test.columns, pd.MultiIndex):
        X_test.columns = ['_'.join(str(c) for c in col) for col in X_test.columns]
    X_test["label"] = y_test_final.values
    X_test = X_test.reset_index(drop=True)
    test_data = X_test
    print(f"Train: {df_ml.shape}, Test réservé: {test_data.shape}")
except NameError:
    print("[WARN] X_test_final non défini — réexécute la cellule 4 d'abord")
    test_data = None

predictor_no_pca = TabularPredictor(label="label", problem_type="multiclass", eval_metric="accuracy", verbosity=2).fit(
    train_data=df_ml, time_limit=300, presets="medium_quality")

print("\n" + "=" * 60)
print("LEADERBOARD (sans PCA)")
print("=" * 60)
if test_data is not None:
    leaderboard = predictor_no_pca.leaderboard(test_data, silent=True)
    print(leaderboard)

    print("\n" + "=" * 60)
    print("ÉVALUATION SUR VRAI TEST SET")
    print("=" * 60)
    perf = predictor_no_pca.evaluate(test_data, auxiliary_metrics=True, silent=True)
    for k, v in perf.items():
        print(f"  {k}: {v:.4f}")

    print("\n" + "=" * 60)
    print("FEATURE IMPORTANCE (top 10)")
    print("=" * 60)
    importance = predictor_no_pca.feature_importance(test_data)
    #print(importance.head(10))
else:
    print("[SKIP] Test non disponible")

No path specified. Models will be saved in: "AutogluonModels/ag-20260610_150419"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #5262-Microsoft Fri Jan 01 08:00:00 PST 2016
CPU Count:          16
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       2.62 GB / 23.91 GB (11.0%)
Disk Space Avail:   414.56 GB / 953.85 GB (43.5%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'


[INFO] Données filtrées
Train: (638, 51), Test réservé: (303, 2654)


Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/mnt/g/ngs/scripts/AutogluonModels/ag-20260610_150419"
Train Data Rows:    638
Train Data Columns: 50
Label Column:       label
Problem Type:       multiclass
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Train Data Class Count: 2
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    2685.50 MB
	Train Data (Original)  Memory Usage: 0.24 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator.


LEADERBOARD (sans PCA)


These features in provided data are not utilized by the predictor and will be ignored: ['count_hsa-let-7a-2-3p', 'count_hsa-let-7a-3p', 'count_hsa-let-7b-3p', 'count_hsa-let-7c-3p', 'count_hsa-let-7c-5p', 'count_hsa-let-7d-3p', 'count_hsa-let-7d-5p', 'count_hsa-let-7e-3p', 'count_hsa-let-7e-5p', 'count_hsa-let-7f-1-3p', 'count_hsa-let-7f-2-3p', 'count_hsa-let-7g-3p', 'count_hsa-let-7i-3p', 'count_hsa-miR-1-3p', 'count_hsa-miR-1-5p', 'count_hsa-miR-100-3p', 'count_hsa-miR-100-5p', 'count_hsa-miR-101-2-5p', 'count_hsa-miR-101-5p', 'count_hsa-miR-10226', 'count_hsa-miR-10392-3p', 'count_hsa-miR-10392-5p', 'count_hsa-miR-10393-3p', 'count_hsa-miR-10393-5p', 'count_hsa-miR-10394-3p', 'count_hsa-miR-10394-5p', 'count_hsa-miR-10395-3p', 'count_hsa-miR-10395-5p', 'count_hsa-miR-10396a-3p', 'count_hsa-miR-10396a-5p', 'count_hsa-miR-10396b-3p', 'count_hsa-miR-10396b-5p', 'count_hsa-miR-10397-3p', 'count_hsa-miR-10397-5p', 'count_hsa-miR-10398-3p', 'count_hsa-miR-10398-5p', 'count_hsa-miR-10399-3

                  model  score_test  score_val eval_metric  pred_time_test  \
0         LightGBMLarge    1.000000   1.000000    accuracy        0.001636   
1              LightGBM    1.000000   1.000000    accuracy        0.001952   
2              CatBoost    1.000000   1.000000    accuracy        0.005555   
3        NeuralNetTorch    1.000000   1.000000    accuracy        0.029446   
4        ExtraTreesEntr    1.000000   1.000000    accuracy        0.085299   
5      RandomForestEntr    1.000000   1.000000    accuracy        0.095465   
6        ExtraTreesGini    1.000000   1.000000    accuracy        0.098896   
7      RandomForestGini    1.000000   1.000000    accuracy        0.099192   
8               XGBoost    0.996700   1.000000    accuracy        0.008307   
9   WeightedEnsemble_L2    0.996700   1.000000    accuracy        0.009991   
10           LightGBMXT    0.993399   1.000000    accuracy        0.001882   
11      NeuralNetFastAI    0.993399   0.992188    accuracy      

	0.46s	= Actual runtime (Completed 5 of 5 shuffle sets)


In [23]:
# ── CELLULE 10 : AutoGluon best_quality (vrai test set) ──
from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings("ignore")

try:
    df_ml = df_tot_filtered.copy()
    print("[INFO] Données filtrées")
except NameError:
    df_ml = df_tot.copy().drop(columns=["label"])

if isinstance(df_ml.columns, pd.MultiIndex):
    df_ml.columns = ['_'.join(str(c) for c in col) for col in df_ml.columns]

df_ml["label"] = encoder.inverse_transform(target)
df_ml = df_ml.reset_index(drop=True)

try:
    X_test = X_test_final.copy()
    if isinstance(X_test.columns, pd.MultiIndex):
        X_test.columns = ['_'.join(str(c) for c in col) for col in X_test.columns]
    X_test["label"] = y_test_final.values
    X_test = X_test.reset_index(drop=True)
    test_data = X_test
    print(f"Train: {df_ml.shape}, Test réservé: {test_data.shape}")
except NameError:
    print("[WARN] X_test_final non défini")
    test_data = None

predictor_best = TabularPredictor(label="label", problem_type="multiclass", eval_metric="accuracy", verbosity=2).fit(
    train_data=df_ml, time_limit=600, presets="best_quality")

print("\n" + "=" * 60)
print("LEADERBOARD (best quality)")
print("=" * 60)
if test_data is not None:
    leaderboard = predictor_best.leaderboard(test_data, silent=True)
    print(leaderboard)

    print("\n" + "=" * 60)
    print("ÉVALUATION SUR VRAI TEST SET")
    print("=" * 60)
    perf = predictor_best.evaluate(test_data, auxiliary_metrics=True, silent=True)
    for k, v in perf.items():
        print(f"  {k}: {v:.4f}")

    print("\n" + "=" * 60)
    print("MATRICE DE CONFUSION")
    print("=" * 60)
    from sklearn.metrics import confusion_matrix
    y_true = test_data["label"]
    y_pred = predictor_best.predict(test_data)
    cm = confusion_matrix(y_true, y_pred)
    print(cm)
else:
    print("[SKIP] Test non disponible")

No path specified. Models will be saved in: "AutogluonModels/ag-20260610_150538"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #5262-Microsoft Fri Jan 01 08:00:00 PST 2016
CPU Count:          16
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       2.99 GB / 23.91 GB (12.5%)
Disk Space Avail:   414.56 GB / 953.85 GB (43.5%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or di

[INFO] Données filtrées
Train: (638, 51), Test réservé: (303, 2654)


(pid=22880) /mnt/g/ngs/scripts/lib/python3.12/site-packages/numpy/_core/getlimits.py:548: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
(pid=22880) This warnings indicates broken support for the dtype!
(pid=22880)   machar = _get_machar(dtype)
(_dystack pid=22880) Running DyStack sub-fit ...
(_dystack pid=22880) Beginning AutoGluon training ... Time limit = 150s
(_dystack pid=22880) AutoGluon will save models to "/mnt/g/ngs/scripts/AutogluonModels/ag-20260610_150538/ds_sub_fit/sub_fit_ho"
(_dystack pid=22880) Train Data Rows:    567
(_dystack pid=22880) Train Data Columns: 50
(_dystack pid=22880) Label Column:       label
(_dystack pid=22880) Problem Type:       multiclass
(_dystack pid=22880) Preprocessing data ...
(_dystack pid=22880) Selected class <--> label mapping:  class 1 = 1, class 0 = 0
(_dystack pid=22880) Train Data Class Count: 2
(

(raylet) Task _ray_fit failed. There are infinite retries remaining, so the task will be retried. Error: Task was killed due to the node running low on memory.

Refer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.


(pid=31054) E0610 17:06:33.318537100   31054 socket_utils_common_posix.cc:224]     check for SO_REUSEPORT: UNKNOWN:Protocol not available {created_time:"2026-06-10T17:06:33.3170739+02:00", errno:92, os_error:"Protocol not available", syscall:"getsockopt(SO_REUSEPORT)"} [repeated 6x across cluster]
(pid=30592) /mnt/g/ngs/scripts/lib/python3.12/site-packages/numpy/_core/getlimits.py:548: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function. [repeated 8x across cluster]
(pid=30592) This warnings indicates broken support for the dtype! [repeated 8x across cluster]
(pid=30592)   machar = _get_machar(dtype) [repeated 8x across cluster]
(pid=30593) E0610 17:06:25.743074700   30593 socket_utils_common_posix.cc:224]     check for SO_REUSEPORT: UNKNOWN:Protocol not available {syscall:"getsockopt(SO_REUSEPORT)", os_error:"Protocol not available", errno:92, create

...

(raylet) Task _ray_fit failed. There are infinite retries remaining, so the task will be retried. Error: Task was killed due to the node running low on memory.

Refer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.


	1.0	 = Validation score   (accuracy)
	22.52s	 = Training   runtime
	0.29s	 = Validation runtime
Fitting model: LightGBM_r131_BAG_L1 ... Training model for up to 128.82s of the 265.99s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=1.03%)
	1.0	 = Validation score   (accuracy)
	3.13s	 = Training   runtime
	0.01s	 = Validation runtime
Fitting model: NeuralNetFastAI_r191_BAG_L1 ... Training model for up to 120.82s of the 257.99s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=0.07%)


(raylet) Task _ray_fit failed. There are infinite retries remaining, so the task will be retried. Error: Task was killed due to the node running low on memory.

Refer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.


	1.0	 = Validation score   (accuracy)
	22.5s	 = Training   runtime
	0.15s	 = Validation runtime
Fitting model: CatBoost_r9_BAG_L1 ... Training model for up to 92.55s of the 229.71s of remaining time.
	Memory not enough to fit 8 folds in parallel. Will train 4 folds in parallel instead (Estimated 11.37% memory usage per fold, 45.49%/80.00% total).
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (4 workers, per: cpus=4, gpus=0, memory=11.37%)
	1.0	 = Validation score   (accuracy)
	20.89s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting model: LightGBM_r96_BAG_L1 ... Training model for up to 68.21s of the 205.37s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=0.31%)
	1.0	 = Validation score   (accuracy)
	1.14s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting model: NeuralNetTorch_r22_BAG_L1 ... Training model for up to 59.94s of the 19


LEADERBOARD (best quality)
                          model  score_test  score_val eval_metric  \
0          LightGBMLarge_BAG_L1    1.000000   0.998433    accuracy   
1           LightGBM_r96_BAG_L1    1.000000   1.000000    accuracy   
2          LightGBM_r131_BAG_L1    1.000000   1.000000    accuracy   
3               LightGBM_BAG_L1    1.000000   0.998433    accuracy   
4          CatBoost_r177_BAG_L1    1.000000   1.000000    accuracy   
5            CatBoost_r9_BAG_L1    1.000000   1.000000    accuracy   
6          CatBoost_r137_BAG_L1    1.000000   1.000000    accuracy   
7               CatBoost_BAG_L1    1.000000   1.000000    accuracy   
8             LightGBMXT_BAG_L1    1.000000   1.000000    accuracy   
9         ExtraTrees_r42_BAG_L1    1.000000   1.000000    accuracy   
10        ExtraTreesEntr_BAG_L1    1.000000   1.000000    accuracy   
11        ExtraTreesGini_BAG_L1    1.000000   1.000000    accuracy   
12      RandomForestEntr_BAG_L1    1.000000   0.998433    accu